# Image Captioning on the Flicker8k Dataset

This notebook provides you with a rough structure for the Mini-Challenge on the topic of Image Captioning from the Deep Learning MSE course. Please make sure to follow the Project Setup steps to ensure reproducable results. Additionally familiarize yourself with the grading structure and project submission guidelines in the _DL-MPW-ImgCaptioning.pdf_ document. There you'll find all the information you should need. 

If you have any questions please contact roman.studer@fhnw.ch

###  Team Name and Members
<span style="color: red;"><strong>TODO:</strong> Add your teamname and add the names of your team members</span>


### Some Guidelines

- You are allowed to use generative models (e.g. ChatGPT, Claude etc.) for code completion or suggestions. However, we require you to perfectly understand and control the code. 
- You are **not** allowed to use the same or similar generative models for providing reflection on the achieved results. __You__ must reflect about what you see and build your own opinion about it.
- You are allowed to create additional python scripts to outsource code snippets to improve readability.

## Project Setup


### 0. Colab environment setup (optional)

*Skip this subsection when running locally in VS Code — it only applies to a fresh Google Colab runtime.* Run these cells top-to-bottom once per session to provision the GPU runtime, pull the repo, install dependencies, stage the Flickr8k dataset, and log into W&B. The steps mirror the standalone `colab_runner.ipynb`.

**Prerequisites (one-time, on the Colab account):**
- Colab Secret `GITLAB_PAT` — GitLab OST personal access token with `read_repository` scope.
- Colab Secret `WANDB_API_KEY` — Weights & Biases API key.
- `flickr8k.zip` uploaded to Drive at `MyDrive/Colab Notebooks/image_captioning_project/data/flickr8k.zip` (must contain an `Images/` folder with the JPGs).

After running these cells the dataset lives at `/content/image_captioning_project/captioning_starter/data/` (matching `dataloader.py` expectations). Set `data_dir` to that path in the "Dataloader, Tokenizer" section below.

#### Verify GPU

In [ ]:
!nvidia-smi

#### Clone the repo

Token is read from Colab Secrets and embedded in the clone URL via `subprocess` so it never appears in cell output or shell history.

In [ ]:
import os, subprocess
from google.colab import userdata

REPO_DIR = "/content/image_captioning_project"
REPO_HOST = "gitlab.ost.ch"
REPO_PATH = "tobias.schuler/image_captioning_project.git"

token = userdata.get("GITLAB_PAT")
if not token:
    raise RuntimeError("Colab Secret 'GITLAB_PAT' is missing or empty.")

clone_url = f"https://oauth2:{token}@{REPO_HOST}/{REPO_PATH}"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"Repo already at {REPO_DIR}; pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1", clone_url, REPO_DIR], check=True)
    print(f"Cloned into {REPO_DIR}")

del token, clone_url  # don't keep the token in memory longer than needed

#### Install / upgrade dependencies

Colab ships older torch/torchvision than `requirements.txt` pins. The other packages (numpy, pandas, matplotlib, Pillow, tqdm) are already preinstalled on Colab. `wandb` isn't, and `nltk` is needed for BLEU.

In [ ]:
%pip install -q -U "torch>=2.6" "torchvision>=0.21" nltk wandb

#### Mount Drive and stage the dataset

Copies `flickr8k.zip` from Drive to local SSD, unzips into a temp folder, and moves the JPGs into `captioning_starter/data/images/` inside the cloned repo. `captions.txt` and `splits/` are already there from the clone, so nothing else needs to be copied.

In [ ]:
import os, shutil, zipfile
from google.colab import drive

drive.mount("/content/drive")

SRC_ZIP     = "/content/drive/MyDrive/Colab Notebooks/image_captioning_project/data/flickr8k.zip"
LOCAL_ZIP   = "/content/flickr8k.zip"
EXTRACT_DIR = "/content/flickr8k_extracted"
IMAGES_DST  = os.path.join(REPO_DIR, "captioning_starter", "data", "images")

os.makedirs(IMAGES_DST, exist_ok=True)
already_populated = any(f.lower().endswith(".jpg") for f in os.listdir(IMAGES_DST))

if already_populated:
    print(f"{IMAGES_DST} already contains JPGs; skipping unzip.")
else:
    if not os.path.exists(SRC_ZIP):
        raise FileNotFoundError(f"Zip not found at {SRC_ZIP}")
    if not os.path.exists(LOCAL_ZIP):
        print(f"Copying {SRC_ZIP} -> {LOCAL_ZIP}")
        shutil.copy(SRC_ZIP, LOCAL_ZIP)
    print(f"Extracting to {EXTRACT_DIR}")
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        z.extractall(EXTRACT_DIR)

    src_images = None
    for candidate in ("Images", "images"):
        p = os.path.join(EXTRACT_DIR, candidate)
        if os.path.isdir(p):
            src_images = p
            break
    if src_images is None:
        raise FileNotFoundError(
            f"No Images/ folder inside the zip; extracted contents: {os.listdir(EXTRACT_DIR)}"
        )

    print(f"Moving JPGs: {src_images} -> {IMAGES_DST}")
    for name in os.listdir(src_images):
        shutil.move(os.path.join(src_images, name), os.path.join(IMAGES_DST, name))

    shutil.rmtree(EXTRACT_DIR, ignore_errors=True)

print(f"\n{IMAGES_DST}: {len(os.listdir(IMAGES_DST))} files")

#### W&B login

In [ ]:
from google.colab import userdata
import wandb

wandb.login(key=userdata.get("WANDB_API_KEY"))

### 1. Create the virtual environment

Create a new environment and install the necessary dependencies listed in `requirements.txt`. Make sure you can import the libraries in the code cell below and that yout pytorch installation can run on your GPU.

In [ ]:
# imports
import copy
import random
import textwrap

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision

import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

from dataloader import (
    build_tokenizer_from_split,
    Flickr8kCaptionsDatasetBase,
    caption_collate_fn,
    load_split_image_ids,
)
from tokenizer import tokenize

print("torch version:", torch.__version__)
print("torchvision version:", torchvision.__version__)

### 2. Download the dataset
The Flickr8k dataset used in this challenge is available under [Flickr 8k Dataset - Kaggle](https://www.kaggle.com/datasets/adityajn105/flickr8k). In the zip folder on Moodle you find the basic structure with `captions.txt` and the `data/splits`-folder, but without images in the `data/ìmages`-folder. Make sure to download all the images and placing them in the folder `data/images/`. 
The train test split is already provided for you and handled by the provided `dataloader.py` script. Test and train split image lists and corresponding captions are located in the `data/splits/` folder. Don't modify this split.

<img src="data_structure.png"/>

## Dataloader, Tokenizer

The repository includes a `Dataset`-class (`Flickr8kCaptionsDatasetBase`) and `DataLoader`'s that are based on this specific structure of the data directory. You can use `create_caption_dataloader` for obtaining instances for the train and test data loaders. You can configure it with the `tokenizer` provided in the repo. The `tokenizer` performs some preprocessing steps of the captions. It also creates a vocabulary from the training data and then provides a mapping from the words to tokens. Note that there are also some special tokens introduced that are important when processing the captions during training or generating the captions during inference:
| Word    | ID | Role |
| --------| ---|----- |
| `<pad>`  | 0 | padding for filling up sequence to a given length in a mini-batch. | 
| `<bos>`  | 1 | beginning of sentence - each tokenized sentence should start with `<bos>`, i.e. also the generated ones. | 
| `<eos>`  | 2 | end of sentence - for fixed length captions will signal that the rest will consist of `<pad>`-tokens only. | 
| `<unk>`  | 3 | unknown - not available in vocabulary | 

Furthermore, specify your transforms used for preprocessing and augmenting the images and pass them to the `create_caption_dataloader`-method.
The mini-batches created by the dataloader have the following structure: `(images, captions, lengths, image_ids, raw_captions)`. Captions are built from the training vocabulary only, and `lengths` may let you mask padding tokens in the loss.


In [ ]:
# Transforms
from torchvision import transforms

IMAGENET_SIZE = 224
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def _convert_to_rgb(image: Image.Image) -> Image.Image:
    return image.convert("RGB")

def build_train_transforms() -> transforms.Compose:
    return transforms.Compose([
            transforms.Lambda(_convert_to_rgb),
            # START - MODIFY / EXTEND THIS PART
            # Augmentation matters on a small dataset (~6k train images) to curb overfitting.
            transforms.RandomResizedCrop(IMAGENET_SIZE, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            # END - MODIFY / EXTEND THIS PART
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

def build_test_transforms() -> transforms.Compose:
    return transforms.Compose([
            transforms.Lambda(_convert_to_rgb),
            transforms.Resize((IMAGENET_SIZE,IMAGENET_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)])

def denormalize_image(image: torch.Tensor) -> torch.Tensor:
    """Undo ImageNet normalization for visualization."""
    mean_tensor = image.new_tensor(IMAGENET_MEAN).view(-1, 1, 1)
    std_tensor = image.new_tensor(IMAGENET_STD).view(-1, 1, 1)
    return (image * std_tensor + mean_tensor).clamp(0, 1)

In [ ]:
# Data + run parameters.
# data_dir points at the local layout `data/` (splits + captions.txt live in the repo;
# the Colab runner stages images into data/images/). Adjust if your layout differs.
data_dir = "data/"

MIN_WORD_FREQ = 5      # words rarer than this collapse to <unk> (keeps the vocab compact)
MAX_LEN = 35           # caption length cap; must match generate(max_len=...) for a fair comparison
BATCH_SIZE = 64

VAL_FRACTION = 0.12    # fraction of TRAIN *images* held out for early stopping / val curves
SEED = 42              # seeds the train/val split and (later) training for reproducibility
NUM_WORKERS = 2        # set to 0 locally if dataloader workers misbehave in the notebook

In [ ]:
# Build the vocabulary from the TRAIN split only (never test) to avoid leakage.
tokenizer = build_tokenizer_from_split(split="train", data_dir=data_dir, min_freq=MIN_WORD_FREQ)

# The assignment ships only a fixed train/test split. To get early stopping and
# validation curves without ever touching the test set, carve a fixed, seeded
# slice of the TRAIN *images* into a validation set. Splitting on images (not
# caption rows) guarantees no image's captions straddle the train/val boundary.
_all_train_images = load_split_image_ids("train", data_dir)
_shuffled = _all_train_images[:]
random.Random(SEED).shuffle(_shuffled)
_n_val = int(round(len(_shuffled) * VAL_FRACTION))
val_image_ids = sorted(_shuffled[:_n_val])
train_image_ids = sorted(_shuffled[_n_val:])


def make_loader(split, transform, caption_sampling, shuffle, image_ids=None):
    """DataLoader over a (possibly image-subsetted) split using the provided tokenizer/transform."""
    dataset = Flickr8kCaptionsDatasetBase(
        split=split,
        data_dir=data_dir,
        tokenizer=tokenizer,
        transform=transform,
        image_ids=image_ids,
        max_len=MAX_LEN,
        caption_sampling=caption_sampling,
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        persistent_workers=NUM_WORKERS > 0,
        collate_fn=caption_collate_fn,
    )


# Train/val: "all" captions per image (~5x samples). Test: "first" to visit each of
# the 1,618 test images exactly once; BLEU is scored against ALL references via
# dataset.captions_map, so the single sampled caption here does not limit evaluation.
train_loader = make_loader("train", build_train_transforms(), "all", shuffle=True, image_ids=train_image_ids)
val_loader = make_loader("train", build_test_transforms(), "all", shuffle=False, image_ids=val_image_ids)
test_loader = make_loader("test", build_test_transforms(), "first", shuffle=False)

# All reference captions per test image, used for corpus BLEU at evaluation time.
test_captions_map = test_loader.dataset.captions_map

print(f"Vocabulary size: {len(tokenizer):,}")
print(f"Train images: {len(train_image_ids):,} | Val images: {len(val_image_ids):,} | Test images: {len(test_loader.dataset.image_ids):,}")
print(f"Train samples (image,caption pairs): {len(train_loader.dataset):,}")
print(f"Val samples (image,caption pairs):   {len(val_loader.dataset):,}")
print(f"Test samples (one per image):        {len(test_loader.dataset):,}")

# Sanity batch (loads images -> run on Colab, or locally once data/images/ is populated).
images, captions, lengths, image_ids, raw_captions = next(iter(train_loader))
print(f"\nBatch image tensor shape:   {tuple(images.shape)}")
print(f"Batch caption tensor shape: {tuple(captions.shape)}")
print(f"Caption lengths in batch:   {lengths.tolist()}")

In [ ]:
num_examples = 12
n = min(num_examples, len(images))

fig, axes = plt.subplots(
    nrows=n,
    ncols=2,
    figsize=(16, 4 * n),
    gridspec_kw={"width_ratios": [1.1, 2.2]},
)

if n == 1:
    axes = [axes]

for idx in range(n):
    image = denormalize_image(images[idx].detach().cpu()).permute(1, 2, 0)

    encoded_caption = captions[idx, : int(lengths[idx])].tolist()
    tokenized_caption = tokenize(raw_captions[idx])
    decoded_caption = tokenizer.decode(encoded_caption, skip_special_tokens=False)

    ax_image, ax_text = axes[idx]
    ax_image.imshow(image)
    ax_image.set_title(image_ids[idx])
    ax_image.axis("off")

    caption_details = "\n\n".join(
        [
            f"Raw caption:\n{textwrap.fill(raw_captions[idx], width=70)}",
            f"Tokenized caption:\n{textwrap.fill(str(tokenized_caption), width=70)}",
            f"Encoded caption:\n{textwrap.fill(str(encoded_caption), width=70)}",
            f"Decoded caption:\n{textwrap.fill(decoded_caption, width=70)}",
        ]
    )

    ax_text.axis("off")
    ax_text.text(
        0,
        1,
        caption_details,
        va="top",
        ha="left",
        fontsize=11,
        family="monospace",
    )

plt.tight_layout()
plt.show()

## Shared Utilities

Use this section for shared config and helper functions reused across all experiments. For example define a common training loop, BLEU computation or visualization helpers here. This way your model models stay directly comparable.


In [ ]:
# Shared config + model-agnostic helpers, reused identically by both models so the
# comparison is fair. Model contract assumed below:
#   forward(images, captions) -> logits (B, T-1, V) aligned to predict captions[:, 1:],
#       or (logits, extras) where extras may carry {"reg_loss": tensor} (e.g. the
#       doubly-stochastic attention penalty for Show-Attend-and-Tell).
#   generate(images, max_len) -> LongTensor (B, L) of generated ids (no <bos>),
#       ending at <eos> and/or padded; decoded with the helpers below.
#   For beam search the model also exposes _beam_init / _beam_step / _beam_reorder.

# ---- device + reproducibility --------------------------------------------------
DEVICE = None
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("device:", DEVICE)


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ---- training / evaluation -----------------------------------------------------
def _forward_logits(model, images, captions):
    """Run forward and normalize the return to (logits, extras_dict)."""
    out = model(images, captions)
    if isinstance(out, tuple):
        return out[0], (out[1] if len(out) > 1 else {})
    return out, {}


def train_one_epoch(model, loader, optimizer, criterion, grad_clip: float = 5.0) -> float:
    """One teacher-forced pass; returns mean per-token CE over non-pad target tokens."""
    model.train()
    total_loss, total_tokens = 0.0, 0
    for images, captions, lengths, image_ids, raw_captions in loader:
        images = images.to(DEVICE)
        captions = captions.to(DEVICE)

        optimizer.zero_grad()
        logits, extras = _forward_logits(model, images, captions)
        targets = captions[:, 1:]                      # next-token targets; <pad> ignored by criterion
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        loss = loss + extras.get("reg_loss", 0.0)      # e.g. attention regularization (0.0 if absent)
        loss.backward()
        if grad_clip is not None:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        n_tok = int((targets != criterion.ignore_index).sum().item())
        total_loss += loss.item() * n_tok
        total_tokens += n_tok
    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_loss(model, loader, criterion) -> float:
    """Mean per-token CE over non-pad target tokens (no augmentation, no grad)."""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for images, captions, lengths, image_ids, raw_captions in loader:
        images = images.to(DEVICE)
        captions = captions.to(DEVICE)
        logits, _ = _forward_logits(model, images, captions)
        targets = captions[:, 1:]
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        n_tok = int((targets != criterion.ignore_index).sum().item())
        total_loss += loss.item() * n_tok
        total_tokens += n_tok
    return total_loss / max(total_tokens, 1)


def compute_perplexity(model, loader, criterion) -> float:
    """exp(mean per-token cross-entropy) over non-pad tokens."""
    return float(np.exp(evaluate_loss(model, loader, criterion)))


def fit(model, train_loader, val_loader, criterion, optimizer,
        num_epochs: int, patience: int = 5, grad_clip: float = 5.0,
        wandb_run=None) -> dict:
    """Train with early stopping on val loss; restore the best weights before returning."""
    history = {"train_loss": [], "val_loss": [], "val_ppl": []}
    best_val, best_state, epochs_no_improve = float("inf"), None, 0

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, grad_clip)
        val_loss = evaluate_loss(model, val_loader, criterion)
        val_ppl = float(np.exp(val_loss))

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_ppl"].append(val_ppl)
        wandb_log_epoch(wandb_run, epoch,
                        {"train/loss": train_loss, "val/loss": val_loss, "val/perplexity": val_ppl})
        print(f"epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f} | val ppl {val_ppl:.2f}")

        if val_loss < best_val - 1e-4:
            best_val, best_state, epochs_no_improve = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"early stopping at epoch {epoch} (best val loss {best_val:.4f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return history


# ---- generation + metrics ------------------------------------------------------
def _ids_to_tokens(ids, tokenizer) -> list[str]:
    """Decode generated ids to word tokens: stop at <eos>, drop all special tokens."""
    tokens = []
    for i in ids:
        token = tokenizer.itos.get(int(i), "<unk>")
        if token == "<eos>":
            break
        if token in tokenizer.special_tokens:
            continue
        tokens.append(token)
    return tokens


@torch.no_grad()
def beam_search_image(model, image: torch.Tensor, tokenizer,
                      beam_size: int = 3, max_len: int = None) -> list:
    """Beam search for a single image (B=1). Returns the highest length-normalized
    token-id sequence (no <bos>; ends at <eos> if found, otherwise truncated).
    Polymorphic on any model implementing _beam_init/_beam_step/_beam_reorder."""
    if max_len is None:
        max_len = MAX_LEN
    K = beam_size
    V = model.vocab_size
    bos, eos = tokenizer.bos_idx, tokenizer.eos_idx
    device = image.device

    # First step from <bos> with K=1; expand to K beams by taking the top-K initial words.
    state = model._beam_init(image)
    prev = torch.tensor([bos], device=device, dtype=torch.long)
    log_probs, state = model._beam_step(state, prev)                    # (1, V)
    top_scores, top_words = log_probs[0].topk(K)                         # (K,)
    sequences = top_words.unsqueeze(1)                                    # (K, 1)
    scores = top_scores
    # Broadcast the single state to K identical beams; they'll diverge from now on.
    state = model._beam_reorder(state, torch.zeros(K, dtype=torch.long, device=device))

    finished = []   # list of (seq_ids_list, raw_score)
    for _ in range(1, max_len):
        live = sequences.size(0)
        if live == 0:
            break
        prev = sequences[:, -1]                                            # (live,)
        log_probs, state = model._beam_step(state, prev)                   # (live, V)
        cand = scores.unsqueeze(1) + log_probs                             # (live, V)
        top_flat_scores, top_flat_idx = cand.reshape(-1).topk(min(K, live * V))
        beam_idx = top_flat_idx // V
        word_idx = top_flat_idx %  V

        sequences = torch.cat([sequences[beam_idx], word_idx.unsqueeze(1)], dim=1)
        scores = top_flat_scores
        state = model._beam_reorder(state, beam_idx)

        is_eos = (word_idx == eos)
        for i in is_eos.nonzero(as_tuple=True)[0].tolist():
            finished.append((sequences[i].tolist(), scores[i].item()))
        keep = ~is_eos
        if not keep.any():
            break
        sequences = sequences[keep]
        scores = scores[keep]
        state = model._beam_reorder(state, keep.nonzero(as_tuple=True)[0])
        if len(finished) >= K:
            break

    # Anything still live at max_len counts as a truncated candidate.
    for i in range(sequences.size(0)):
        finished.append((sequences[i].tolist(), scores[i].item()))

    if not finished:
        return [eos]
    best_seq, _ = max(finished, key=lambda x: x[1] / max(len(x[0]), 1))   # length-normalized
    return best_seq


@torch.no_grad()
def generate_captions(model, loader, tokenizer, max_len: int = MAX_LEN,
                      decoder: str = "greedy", beam_size: int = 3) -> dict:
    """Decode every image in the loader. Returns {image_id: [word tokens]}.
    decoder='greedy' uses model.generate (batched); decoder='beam' runs per-image."""
    if decoder not in ("greedy", "beam"):
        raise ValueError(f"decoder must be 'greedy' or 'beam', got {decoder!r}")
    model.eval()
    results = {}
    for images, captions, lengths, image_ids, raw_captions in loader:
        images = images.to(DEVICE)
        if decoder == "greedy":
            seqs = model.generate(images, max_len=max_len).cpu().tolist()
            for image_id, ids in zip(image_ids, seqs):
                results[image_id] = _ids_to_tokens(ids, tokenizer)
        else:
            for i in range(images.size(0)):
                ids = beam_search_image(model, images[i:i+1], tokenizer,
                                        beam_size=beam_size, max_len=max_len)
                results[image_ids[i]] = _ids_to_tokens(ids, tokenizer)
    return results


def corpus_bleu_scores(hypotheses: dict, captions_map: dict, tokenizer) -> dict:
    """Corpus BLEU-1..4 of {image_id: [tokens]} hypotheses vs all references per image."""
    smooth = SmoothingFunction().method1
    list_of_references, list_of_hypotheses = [], []
    for image_id, hyp_tokens in hypotheses.items():
        references = [tokenize(caption) for caption in captions_map[image_id]]
        list_of_references.append(references)
        list_of_hypotheses.append(hyp_tokens)

    scores = {}
    for n in range(1, 5):
        weights = tuple([1.0 / n] * n + [0.0] * (4 - n))
        scores[f"bleu{n}"] = corpus_bleu(
            list_of_references, list_of_hypotheses,
            weights=weights, smoothing_function=smooth,
        )
    return scores


def evaluate_model(model, test_loader, tokenizer, criterion,
                   decoder: str = "greedy", beam_size: int = 3) -> tuple[dict, dict]:
    """Identical eval used for both models: perplexity + corpus BLEU-1..4.
    decoder/beam_size let the same helper run greedy or beam decoding."""
    perplexity = compute_perplexity(model, test_loader, criterion)
    hypotheses = generate_captions(model, test_loader, tokenizer, max_len=MAX_LEN,
                                   decoder=decoder, beam_size=beam_size)
    bleu = corpus_bleu_scores(hypotheses, test_loader.dataset.captions_map, tokenizer)
    metrics = {"perplexity": perplexity, **bleu}
    return metrics, hypotheses


# ---- visualization -------------------------------------------------------------
def plot_curves(history: dict, title: str = "Training curves") -> None:
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].set_title("Loss"); axes[0].legend()
    axes[1].plot(epochs, history["val_ppl"], color="tab:green")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("perplexity"); axes[1].set_title("Validation perplexity")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


@torch.no_grad()
def show_predictions(model, loader, tokenizer, num_examples: int = 6, max_len: int = MAX_LEN) -> None:
    """Display a batch of images with the model's greedy caption and one reference."""
    model.eval()
    images, captions, lengths, image_ids, raw_captions = next(iter(loader))
    seqs = model.generate(images.to(DEVICE), max_len=max_len).cpu().tolist()
    captions_map = loader.dataset.captions_map

    n = min(num_examples, len(image_ids))
    fig, axes = plt.subplots(n, 1, figsize=(5, 4 * n))
    if n == 1:
        axes = [axes]
    for i in range(n):
        image = denormalize_image(images[i].cpu()).permute(1, 2, 0)
        prediction = " ".join(_ids_to_tokens(seqs[i], tokenizer))
        reference = captions_map[image_ids[i]][0]
        axes[i].imshow(image)
        axes[i].axis("off")
        axes[i].set_title(f"pred: {prediction}\nref:  {reference}", fontsize=9, loc="left")
    plt.tight_layout()
    plt.show()


# ---- experiment tracking (Weights & Biases; optional, lazily imported) ----------
def init_wandb(model_name: str, config: dict, project: str = "dl-mpw-image-captioning",
               group: str | None = None, mode: str | None = None):
    """Start a W&B run. Returns None (and prints) if wandb is unavailable, so the
    notebook still runs end-to-end without tracking."""
    try:
        import wandb
    except ImportError:
        print("wandb not installed - skipping experiment tracking.")
        return None
    return wandb.init(
        project=project, name=model_name, group=group or model_name,
        config=config, mode=mode, reinit=True,
    )


def wandb_log_epoch(run, epoch: int, metrics: dict) -> None:
    if run is not None:
        run.log({**metrics, "epoch": epoch})


def finish_wandb(run) -> None:
    if run is not None:
        run.finish()

## Model 1 - Show and Tell

Implement the model described in the [_Show and Tell_](https://arxiv.org/abs/1411.4555) paper . Record the training behavior, inspect a few generated captions on held-out images, and report BLEU on your fixed evaluation subset.

In [ ]:
# Show-and-Tell hyperparameters.
# EMBED_DIM/DECODER_DIM are kept identical in Model 2 so the two architectures differ
# only by the attention mechanism, not by raw capacity.
ENCODER_DIM = 512        # ResNet18 avgpool output
EMBED_DIM = 256
DECODER_DIM = 512
DROPOUT = 0.5
FREEZE_ENCODER = True    # ImageNet-pretrained features; ~6k train images don't justify finetuning the backbone

In [ ]:
class ShowAndTellEncoder(nn.Module):
    """ResNet18 truncated at avgpool -> (B, 512) image feature, no classifier head."""

    def __init__(self, freeze: bool = True):
        super().__init__()
        backbone = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.DEFAULT
        )
        backbone.fc = nn.Identity()           # avgpool already produces (B, 512)
        self.backbone = backbone
        self._frozen = freeze
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False
            self.backbone.eval()

    def train(self, mode: bool = True):
        # Keep frozen BatchNorm in eval so its running stats don't drift on our small dataset.
        super().train(mode)
        if self._frozen:
            self.backbone.eval()
        return self

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        return self.backbone(images)          # (B, 512)


class ShowAndTellCaptioner(nn.Module):
    """Show and Tell (Vinyals et al. 2014): the image embedding is fed as the LSTM's
    first input, followed by teacher-forced word embeddings."""

    def __init__(self, encoder: ShowAndTellEncoder, vocab_size: int,
                 encoder_dim: int = ENCODER_DIM, embed_dim: int = EMBED_DIM,
                 decoder_dim: int = DECODER_DIM, dropout: float = DROPOUT,
                 bos_idx: int = 1, eos_idx: int = 2, pad_idx: int = 0):
        super().__init__()
        self.encoder = encoder
        self.vocab_size = vocab_size
        self.bos_idx, self.eos_idx, self.pad_idx = bos_idx, eos_idx, pad_idx

        self.img_proj = nn.Linear(encoder_dim, embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, decoder_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(decoder_dim, vocab_size)

    def forward(self, images: torch.Tensor, captions: torch.Tensor) -> torch.Tensor:
        # Inputs:  [img_emb, emb(<bos>), emb(w_1), ..., emb(w_{T-2})]  -> T LSTM steps.
        # Targets: [w_1, w_2, ..., w_{T-2}, <eos>/<pad>]              -> T-1 next-token preds.
        # We drop the image-step output so logits[:, t] predicts captions[:, t+1].
        img_feat = self.encoder(images)                         # (B, 512)
        img_emb = self.img_proj(img_feat).unsqueeze(1)          # (B, 1, E)
        word_emb = self.embedding(captions[:, :-1])             # (B, T-1, E)
        inputs = torch.cat([img_emb, word_emb], dim=1)          # (B, T, E)
        outputs, _ = self.lstm(inputs)                          # (B, T, D)
        outputs = self.dropout(outputs[:, 1:])                  # drop image-step output
        return self.fc(outputs)                                 # (B, T-1, V)

    @torch.no_grad()
    def generate(self, images: torch.Tensor, max_len: int = 35) -> torch.Tensor:
        """Greedy decode. Returns (B, max_len) ids: no <bos>, ending at <eos>, padded after."""
        device = images.device
        batch = images.size(0)

        img_feat = self.encoder(images)
        img_emb = self.img_proj(img_feat).unsqueeze(1)          # (B, 1, E)
        _, state = self.lstm(img_emb)                            # initial (h, c) after seeing the image

        prev = torch.full((batch, 1), self.bos_idx, dtype=torch.long, device=device)
        generated = torch.full((batch, max_len), self.pad_idx, dtype=torch.long, device=device)
        finished = torch.zeros(batch, dtype=torch.bool, device=device)

        for t in range(max_len):
            out, state = self.lstm(self.embedding(prev), state)  # (B, 1, D)
            next_tok = self.fc(out.squeeze(1)).argmax(dim=-1)    # (B,)
            # Once a sample has emitted <eos>, keep its slots as <pad>.
            next_tok = torch.where(finished, torch.full_like(next_tok, self.pad_idx), next_tok)
            generated[:, t] = next_tok
            finished = finished | (next_tok == self.eos_idx)
            if bool(finished.all()):
                break
            prev = next_tok.unsqueeze(1)
        return generated

    # ---- beam-search step interface (used by shared utils beam_search_image) -----
    # The state dict holds the LSTM's (h, c) where each has shape (num_layers=1, K, D).
    @torch.no_grad()
    def _beam_init(self, image: torch.Tensor) -> dict:
        """Single image -> initial state after the LSTM has seen the image."""
        img_feat = self.encoder(image)
        img_emb = self.img_proj(img_feat).unsqueeze(1)           # (1, 1, E)
        _, (h, c) = self.lstm(img_emb)                            # h, c: (1, 1, D)
        return {"h": h, "c": c}

    @torch.no_grad()
    def _beam_step(self, state: dict, prev_tok: torch.Tensor):
        """state['h'/'c']: (1, K, D). prev_tok: (K,). Returns (log_probs (K, V), new_state)."""
        emb = self.embedding(prev_tok).unsqueeze(1)              # (K, 1, E)
        out, (h, c) = self.lstm(emb, (state["h"], state["c"]))   # out (K, 1, D)
        log_probs = F.log_softmax(self.fc(out.squeeze(1)), dim=-1)
        return log_probs, {"h": h, "c": c}

    @staticmethod
    def _beam_reorder(state: dict, indices: torch.Tensor) -> dict:
        # LSTM h/c have shape (num_layers=1, K, D); index along the beam dim (1).
        return {"h": state["h"][:, indices], "c": state["c"][:, indices]}

In [ ]:
# Build, train, and evaluate Show and Tell.
# Optimizer skips frozen encoder params; weight decay + dropout + early stopping = regularization.
LR_M1 = 4e-4
WEIGHT_DECAY_M1 = 1e-5
NUM_EPOCHS_M1 = 20
PATIENCE_M1 = 4

set_seed(SEED)
encoder_m1 = ShowAndTellEncoder(freeze=FREEZE_ENCODER).to(DEVICE)
model_m1 = ShowAndTellCaptioner(
    encoder=encoder_m1,
    vocab_size=len(tokenizer),
    encoder_dim=ENCODER_DIM,
    embed_dim=EMBED_DIM,
    decoder_dim=DECODER_DIM,
    dropout=DROPOUT,
    bos_idx=tokenizer.bos_idx,
    eos_idx=tokenizer.eos_idx,
    pad_idx=tokenizer.pad_idx,
).to(DEVICE)

trainable_m1 = [p for p in model_m1.parameters() if p.requires_grad]
print(f"trainable params: {sum(p.numel() for p in trainable_m1):,}")

criterion_m1 = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)
optimizer_m1 = torch.optim.Adam(trainable_m1, lr=LR_M1, weight_decay=WEIGHT_DECAY_M1)

config_m1 = {
    "model": "show-and-tell", "encoder": "resnet18",
    "encoder_dim": ENCODER_DIM, "embed_dim": EMBED_DIM, "decoder_dim": DECODER_DIM,
    "dropout": DROPOUT, "freeze_encoder": FREEZE_ENCODER,
    "lr": LR_M1, "weight_decay": WEIGHT_DECAY_M1,
    "batch_size": BATCH_SIZE, "max_len": MAX_LEN,
    "num_epochs": NUM_EPOCHS_M1, "patience": PATIENCE_M1,
}
wandb_run_m1 = init_wandb("show-and-tell", config_m1, group="show-and-tell")

history_m1 = fit(
    model_m1, train_loader, val_loader,
    criterion=criterion_m1, optimizer=optimizer_m1,
    num_epochs=NUM_EPOCHS_M1, patience=PATIENCE_M1,
    wandb_run=wandb_run_m1,
)
plot_curves(history_m1, title="Show and Tell - training")

# Final test evaluation (greedy decoding, BLEU vs all ~5 refs per image).
metrics_m1, hypotheses_m1 = evaluate_model(model_m1, test_loader, tokenizer, criterion_m1)
print("\nShow-and-Tell test metrics:")
for k, v in metrics_m1.items():
    print(f"  {k}: {v:.4f}")
if wandb_run_m1 is not None:
    wandb_run_m1.log({f"test/{k}": v for k, v in metrics_m1.items()})

# Qualitative: greedy captions on a handful of test images.
show_predictions(model_m1, test_loader, tokenizer, num_examples=6)

finish_wandb(wandb_run_m1)

Briefly summarize what happened during training, which captions looked convincing or weak, and what the BLEU score does or does not capture for this model.


<span style="color: red;"><strong>TODO:</strong> Write your reflection on Model 1 here.</span>


## Model 2 - Show, Attend and Tell

Now implement the continuation of the Show and Tell paper. An attention-based captioning model described in the paper [_Show, Attend and Tell_](https://arxiv.org/abs/1502.03044). Use the same test data subset and BLEU setup as before, then compare how attention changes the training behavior and generated captions.


In [ ]:
# Show-Attend-and-Tell hyperparameters.
# ENCODER_DIM/EMBED_DIM/DECODER_DIM/DROPOUT/FREEZE_ENCODER are re-stated with the SAME
# values as Model 1 so the two architectures differ only by the attention mechanism.
ENCODER_DIM = 512        # ResNet18 layer4 channel dim (spatial map is 7x7 -> 49 locations)
EMBED_DIM = 256
DECODER_DIM = 512
DROPOUT = 0.5
FREEZE_ENCODER = True
ATTENTION_DIM = 256      # Bahdanau attention projection dim

In [ ]:
class ShowAttendTellEncoder(nn.Module):
    """ResNet18 truncated *before* avgpool -> spatial feature map (B, 512, 7, 7),
    reshaped to (B, 49, 512) so the decoder can attend over spatial locations."""

    def __init__(self, freeze: bool = True):
        super().__init__()
        backbone = torchvision.models.resnet18(
            weights=torchvision.models.ResNet18_Weights.DEFAULT
        )
        # Drop avgpool and fc; the remaining stack ends at layer4 -> (B, 512, 7, 7) at 224x224.
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        self._frozen = freeze
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False
            self.backbone.eval()

    def train(self, mode: bool = True):
        super().train(mode)
        if self._frozen:
            self.backbone.eval()
        return self

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        feat = self.backbone(images)                           # (B, 512, 7, 7)
        B, C, H, W = feat.shape
        return feat.view(B, C, H * W).transpose(1, 2)          # (B, 49, 512)


class AdditiveAttention(nn.Module):
    """Bahdanau-style additive attention over a set of encoder feature vectors.
    score_i = v^T tanh(W_e e_i + W_d h);  alpha = softmax(scores);  context = sum(alpha_i e_i)."""

    def __init__(self, encoder_dim: int, decoder_dim: int, attention_dim: int):
        super().__init__()
        self.enc_att = nn.Linear(encoder_dim, attention_dim)
        self.dec_att = nn.Linear(decoder_dim, attention_dim)
        self.full_att = nn.Linear(attention_dim, 1)

    def forward(self, encoder_out: torch.Tensor, decoder_hidden: torch.Tensor):
        # encoder_out:    (B, P, encoder_dim)
        # decoder_hidden: (B, decoder_dim)
        att_enc = self.enc_att(encoder_out)                            # (B, P, A)
        att_dec = self.dec_att(decoder_hidden).unsqueeze(1)             # (B, 1, A)
        scores = self.full_att(torch.tanh(att_enc + att_dec))           # (B, P, 1)
        alpha = F.softmax(scores.squeeze(-1), dim=1)                    # (B, P)
        context = (alpha.unsqueeze(-1) * encoder_out).sum(dim=1)        # (B, encoder_dim)
        return context, alpha


class ShowAttendTellCaptioner(nn.Module):
    """Show, Attend and Tell (Xu et al. 2015), soft-attention variant.

    Per step: attention(encoder_out, h) -> context; sigmoid gate beta(h) scales the
    context; LSTMCell consumes [prev_embedding, gated_context]; fc -> vocab logits.
    Initial (h0, c0) are predicted from the mean encoder feature via two tanh-MLPs.
    """

    def __init__(self, encoder: ShowAttendTellEncoder, vocab_size: int,
                 encoder_dim: int = ENCODER_DIM, embed_dim: int = EMBED_DIM,
                 decoder_dim: int = DECODER_DIM, attention_dim: int = ATTENTION_DIM,
                 dropout: float = DROPOUT, alpha_c: float = 1.0,
                 bos_idx: int = 1, eos_idx: int = 2, pad_idx: int = 0):
        super().__init__()
        self.encoder = encoder
        self.vocab_size = vocab_size
        self.bos_idx, self.eos_idx, self.pad_idx = bos_idx, eos_idx, pad_idx
        self.encoder_dim = encoder_dim
        self.alpha_c = alpha_c                                          # doubly-stochastic reg strength

        self.attention = AdditiveAttention(encoder_dim, decoder_dim, attention_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.init_h = nn.Linear(encoder_dim, decoder_dim)               # f_init_h (paper)
        self.init_c = nn.Linear(encoder_dim, decoder_dim)               # f_init_c (paper)
        self.f_beta = nn.Linear(decoder_dim, encoder_dim)               # sigmoid gate on the context
        self.lstm_cell = nn.LSTMCell(embed_dim + encoder_dim, decoder_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(decoder_dim, vocab_size)

    def _init_hidden(self, encoder_out: torch.Tensor):
        mean_feat = encoder_out.mean(dim=1)                              # (B, encoder_dim)
        return torch.tanh(self.init_h(mean_feat)), torch.tanh(self.init_c(mean_feat))

    def forward(self, images: torch.Tensor, captions: torch.Tensor):
        encoder_out = self.encoder(images)                               # (B, P, encoder_dim)
        h, c = self._init_hidden(encoder_out)
        embeddings = self.embedding(captions[:, :-1])                    # (B, T-1, E)
        T_in = embeddings.size(1)

        logits_steps, alphas_steps = [], []
        for t in range(T_in):
            context, alpha = self.attention(encoder_out, h)              # (B, ED), (B, P)
            gated_context = torch.sigmoid(self.f_beta(h)) * context      # (B, ED)
            lstm_in = torch.cat([embeddings[:, t], gated_context], dim=1)
            h, c = self.lstm_cell(lstm_in, (h, c))
            logits_steps.append(self.fc(self.dropout(h)))                # (B, V)
            alphas_steps.append(alpha)
        logits = torch.stack(logits_steps, dim=1)                        # (B, T-1, V)
        alphas = torch.stack(alphas_steps, dim=1)                        # (B, T-1, P)

        # Doubly-stochastic attention regularization (Xu et al. 2015 eq. 14).
        # We mask out timesteps whose target is <pad> so the penalty doesn't depend on
        # how much each caption was padded; sum over the *real* steps should be ~= 1
        # per spatial cell.
        target_mask = (captions[:, 1:] != self.pad_idx).float()           # (B, T-1)
        masked_alpha_sums = (alphas * target_mask.unsqueeze(-1)).sum(dim=1)  # (B, P)
        reg_loss = self.alpha_c * ((1.0 - masked_alpha_sums) ** 2).mean()

        return logits, {"reg_loss": reg_loss, "alphas": alphas}

    @torch.no_grad()
    def generate(self, images: torch.Tensor, max_len: int = 35,
                 return_alphas: bool = False):
        """Greedy decode. Returns (B, max_len) ids; with return_alphas=True also
        returns (B, t_used, P) attention weights for visualization (Additional Task)."""
        device = images.device
        encoder_out = self.encoder(images)                               # (B, P, encoder_dim)
        B = encoder_out.size(0)
        h, c = self._init_hidden(encoder_out)

        prev = torch.full((B,), self.bos_idx, dtype=torch.long, device=device)
        generated = torch.full((B, max_len), self.pad_idx, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        alphas_steps = []

        for t in range(max_len):
            emb = self.embedding(prev)                                   # (B, E)
            context, alpha = self.attention(encoder_out, h)              # (B, ED), (B, P)
            gated_context = torch.sigmoid(self.f_beta(h)) * context
            h, c = self.lstm_cell(torch.cat([emb, gated_context], dim=1), (h, c))
            next_tok = self.fc(h).argmax(dim=-1)                         # (B,)
            next_tok = torch.where(finished, torch.full_like(next_tok, self.pad_idx), next_tok)
            generated[:, t] = next_tok
            alphas_steps.append(alpha)
            finished = finished | (next_tok == self.eos_idx)
            if bool(finished.all()):
                break
            prev = next_tok

        if return_alphas:
            return generated, torch.stack(alphas_steps, dim=1)            # (B, t_used, P)
        return generated

    # ---- beam-search step interface (used by shared utils beam_search_image) -----
    # The state dict holds encoder_out (B=K, P, ED) and h, c (K, D); both are reordered
    # across beams by _beam_reorder.
    @torch.no_grad()
    def _beam_init(self, image: torch.Tensor) -> dict:
        encoder_out = self.encoder(image)                                 # (1, P, ED)
        h, c = self._init_hidden(encoder_out)                              # (1, D), (1, D)
        return {"encoder_out": encoder_out, "h": h, "c": c}

    @torch.no_grad()
    def _beam_step(self, state: dict, prev_tok: torch.Tensor):
        emb = self.embedding(prev_tok)                                    # (K, E)
        context, _ = self.attention(state["encoder_out"], state["h"])     # (K, ED)
        gated = torch.sigmoid(self.f_beta(state["h"])) * context
        h, c = self.lstm_cell(torch.cat([emb, gated], dim=1),
                              (state["h"], state["c"]))
        log_probs = F.log_softmax(self.fc(h), dim=-1)                     # (K, V)
        return log_probs, {"encoder_out": state["encoder_out"], "h": h, "c": c}

    @staticmethod
    def _beam_reorder(state: dict, indices: torch.Tensor) -> dict:
        return {
            "encoder_out": state["encoder_out"][indices],
            "h": state["h"][indices],
            "c": state["c"][indices],
        }

In [ ]:
# Build, train, and evaluate Show, Attend and Tell.
# Same optimizer/loss/eval protocol as Model 1 so the two runs are directly comparable.
LR_M2 = 4e-4
WEIGHT_DECAY_M2 = 1e-5
NUM_EPOCHS_M2 = 20
PATIENCE_M2 = 4
ALPHA_C = 1.0                # doubly-stochastic attention regularization weight (paper value)

set_seed(SEED)
encoder_m2 = ShowAttendTellEncoder(freeze=FREEZE_ENCODER).to(DEVICE)
model_m2 = ShowAttendTellCaptioner(
    encoder=encoder_m2,
    vocab_size=len(tokenizer),
    encoder_dim=ENCODER_DIM,
    embed_dim=EMBED_DIM,
    decoder_dim=DECODER_DIM,
    attention_dim=ATTENTION_DIM,
    dropout=DROPOUT,
    alpha_c=ALPHA_C,
    bos_idx=tokenizer.bos_idx,
    eos_idx=tokenizer.eos_idx,
    pad_idx=tokenizer.pad_idx,
).to(DEVICE)

trainable_m2 = [p for p in model_m2.parameters() if p.requires_grad]
print(f"trainable params: {sum(p.numel() for p in trainable_m2):,}")

criterion_m2 = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)
optimizer_m2 = torch.optim.Adam(trainable_m2, lr=LR_M2, weight_decay=WEIGHT_DECAY_M2)

config_m2 = {
    "model": "show-attend-tell", "encoder": "resnet18-layer4",
    "encoder_dim": ENCODER_DIM, "embed_dim": EMBED_DIM, "decoder_dim": DECODER_DIM,
    "attention_dim": ATTENTION_DIM, "dropout": DROPOUT, "freeze_encoder": FREEZE_ENCODER,
    "alpha_c": ALPHA_C,
    "lr": LR_M2, "weight_decay": WEIGHT_DECAY_M2,
    "batch_size": BATCH_SIZE, "max_len": MAX_LEN,
    "num_epochs": NUM_EPOCHS_M2, "patience": PATIENCE_M2,
}
wandb_run_m2 = init_wandb("show-attend-tell", config_m2, group="show-attend-tell")

history_m2 = fit(
    model_m2, train_loader, val_loader,
    criterion=criterion_m2, optimizer=optimizer_m2,
    num_epochs=NUM_EPOCHS_M2, patience=PATIENCE_M2,
    wandb_run=wandb_run_m2,
)
plot_curves(history_m2, title="Show, Attend and Tell - training")

# Identical test eval (greedy decoding, BLEU vs all references) -> directly comparable to Model 1.
metrics_m2, hypotheses_m2 = evaluate_model(model_m2, test_loader, tokenizer, criterion_m2)
print("\nShow-Attend-Tell test metrics:")
for k, v in metrics_m2.items():
    print(f"  {k}: {v:.4f}")
if wandb_run_m2 is not None:
    wandb_run_m2.log({f"test/{k}": v for k, v in metrics_m2.items()})

show_predictions(model_m2, test_loader, tokenizer, num_examples=6)
finish_wandb(wandb_run_m2)

# Side-by-side comparison table (printed inline; PDF report can pull from here).
if "metrics_m1" in globals():
    print("\nModel comparison (test set):")
    print(f"{'metric':<14}{'show-and-tell':>16}{'show-attend-tell':>20}")
    for key in ("perplexity", "bleu1", "bleu2", "bleu3", "bleu4"):
        print(f"{key:<14}{metrics_m1[key]:>16.4f}{metrics_m2[key]:>20.4f}")

Briefly summarize how this model behaved during training, where attention seemed to help or fail, and how the BLEU score compares to Model 2.


<span style="color: red;"><strong>TODO:</strong> Write your reflection on Model 2 here.</span>


## Additional Task 1: Beam Search vs Greedy Decoding

**Hypothesis.** Greedy decoding commits to the locally most-probable word at each step and can get stuck on a plausible-but-suboptimal prefix. Beam search keeps the top-`k` partial sequences alive and so explores a wider tree before committing. We expect beam search (with `beam_size in {3, 5}`) to give **higher BLEU than greedy for both models**, with diminishing returns past `beam_size=5`. The gain should be at least as large for `Show, Attend and Tell` as for `Show and Tell`, because attention provides richer per-step context and so the second-best prefix often carries genuinely different information.

**Setup.** We re-run the same fixed test set, the same BLEU-1..4 protocol, and the same trained checkpoints from Sections B and C; the only change is the decoding strategy. Per-image beam search is run by the shared `beam_search_image` helper via each model's `_beam_init / _beam_step / _beam_reorder` interface, with length-normalized scoring.

In [ ]:
# Decode the full test set with beam_size in {3, 5} for both models, then tabulate
# greedy vs beam BLEU-1..4. Greedy scores are already in metrics_m1 / metrics_m2 from
# Sections B and C, so we don't recompute them here.
BEAM_SIZES = [3, 5]

beam_results = {}      # (model_name, beam_size) -> {"perplexity": ppl, "bleu1": ..., ...}
beam_hypotheses = {}

models_to_eval = [
    ("show-and-tell",    model_m1, criterion_m1),
    ("show-attend-tell", model_m2, criterion_m2),
]

for name, model, criterion in models_to_eval:
    for k in BEAM_SIZES:
        print(f"\nDecoding {name} with beam_size={k} (1,618 images) ...")
        metrics, hyps = evaluate_model(model, test_loader, tokenizer, criterion,
                                       decoder="beam", beam_size=k)
        beam_results[(name, k)] = metrics
        beam_hypotheses[(name, k)] = hyps
        print(f"  ppl {metrics['perplexity']:.2f} | "
              f"B1 {metrics['bleu1']:.4f}  B2 {metrics['bleu2']:.4f}  "
              f"B3 {metrics['bleu3']:.4f}  B4 {metrics['bleu4']:.4f}")

# Comparison table: each model with greedy vs beam-3 vs beam-5.
print(f"\n{'model':<18}{'decoder':<10}{'B1':>9}{'B2':>9}{'B3':>9}{'B4':>9}")
print("-" * 64)
for name, _, _ in models_to_eval:
    greedy = metrics_m1 if name == "show-and-tell" else metrics_m2
    print(f"{name:<18}{'greedy':<10}"
          f"{greedy['bleu1']:>9.4f}{greedy['bleu2']:>9.4f}"
          f"{greedy['bleu3']:>9.4f}{greedy['bleu4']:>9.4f}")
    for k in BEAM_SIZES:
        m = beam_results[(name, k)]
        print(f"{name:<18}{f'beam-{k}':<10}"
              f"{m['bleu1']:>9.4f}{m['bleu2']:>9.4f}"
              f"{m['bleu3']:>9.4f}{m['bleu4']:>9.4f}")

# A single W&B summary run keeps the beam-vs-greedy comparison in one place.
summary_run = init_wandb("beam-vs-greedy", {"beam_sizes": BEAM_SIZES}, group="decoding-comparison")
if summary_run is not None:
    for name, _, _ in models_to_eval:
        greedy = metrics_m1 if name == "show-and-tell" else metrics_m2
        for metric, value in greedy.items():
            summary_run.log({f"{name}/greedy/{metric}": value})
        for k in BEAM_SIZES:
            for metric, value in beam_results[(name, k)].items():
                summary_run.log({f"{name}/beam{k}/{metric}": value})
    finish_wandb(summary_run)

# A few side-by-side qualitative samples (greedy vs beam) for the report.
N_QUAL = 4
sample_images, _, _, sample_ids, _ = next(iter(test_loader))
print(f"\nQualitative comparison (first {N_QUAL} test images):")

greedy_per_model = {}
with torch.no_grad():
    for name, model, _ in models_to_eval:
        model.eval()
        greedy_ids = model.generate(sample_images[:N_QUAL].to(DEVICE), max_len=MAX_LEN).cpu().tolist()
        greedy_per_model[name] = [_ids_to_tokens(ids, tokenizer) for ids in greedy_ids]

for i in range(min(N_QUAL, len(sample_ids))):
    img_id = sample_ids[i]
    ref = test_captions_map[img_id][0]
    print(f"\n[{img_id}]  ref: {ref}")
    for name, _, _ in models_to_eval:
        beam_tokens = beam_hypotheses[(name, BEAM_SIZES[-1])][img_id]
        print(f"  {name:<18} greedy:        {' '.join(greedy_per_model[name][i])}")
        print(f"  {name:<18} beam-{BEAM_SIZES[-1]}:        {' '.join(beam_tokens)}")

Compare beam-3 and beam-5 against greedy: did BLEU improve as predicted, and by similar margins for both architectures? Comment on caption-level effects that beam search introduced or fixed (repetitions, drift, generic phrasing), and on whether the compute cost is justified by the gain.

<span style="color: red;"><strong>TODO:</strong> Write your reflection on the beam-vs-greedy experiment here.</span>

## Additional Task 2: Attention Map Visualization

**Hypothesis.** If `Show, Attend and Tell` has learned anything useful about *where* to look, its per-word attention `alpha` (a distribution over the 49 spatial cells of the `7x7` `layer4` feature map) should concentrate on the image region that is semantically relevant to the word being emitted - faces when saying *man*/*woman*, hands when saying *holding*, and so on. We expect:

- Clean, peaky maps on success cases (correct content word -> clear spatial focus).
- Diffuse or wrong-region maps on failure cases (hallucinated words, missing objects, generic captions like *a man is standing*).

**Setup.** We pick a handful of test images, greedy-decode each with `model_m2.generate(..., return_alphas=True)`, upsample each 7x7 alpha map bilinearly to 224x224, and overlay it on the `denormalize_image()` view. Each row is one image; columns are the original image followed by one panel per generated word.

In [ ]:
# Attention visualization for Show, Attend and Tell.
# We generate captions with the model's per-step alphas, then for each non-special
# generated word we overlay its (upsampled) 7x7 attention map on the source image.
NUM_VIS = 4   # show this many images; pick a mix of likely-success and likely-failure cases below

# Pull a batch from test_loader and pick the first NUM_VIS images. To curate
# success/failure cases, edit the slice or filter by image_id.
sample_images, _, _, sample_ids, _ = next(iter(test_loader))
vis_images = sample_images[:NUM_VIS]
vis_ids = sample_ids[:NUM_VIS]

model_m2.eval()
with torch.no_grad():
    gen_ids, gen_alphas = model_m2.generate(vis_images.to(DEVICE),
                                            max_len=MAX_LEN, return_alphas=True)
gen_ids = gen_ids.cpu()
gen_alphas = gen_alphas.cpu()


def _upsample_alpha(alpha_49: torch.Tensor, size: int = IMAGENET_SIZE) -> np.ndarray:
    """Bilinearly upsample a 49-vector (=7x7) alpha map to (size, size)."""
    grid = alpha_49.view(1, 1, 7, 7)
    up = F.interpolate(grid, size=(size, size), mode="bilinear", align_corners=False)
    return up.squeeze().numpy()


vis_run = init_wandb("attention-viz", {"num_images": NUM_VIS}, group="attention-visualization")

for i in range(NUM_VIS):
    image_denorm = denormalize_image(vis_images[i]).permute(1, 2, 0).numpy()

    # Walk the generated sequence, collecting (token, alpha_map) only for non-special words.
    seq = gen_ids[i].tolist()
    overlays = []
    for t, token_id in enumerate(seq):
        if t >= gen_alphas.size(1):
            break                                  # generation stopped before this t
        token = tokenizer.itos.get(int(token_id), "<unk>")
        if token == "<eos>":
            break
        if token in tokenizer.special_tokens:
            continue
        overlays.append((token, _upsample_alpha(gen_alphas[i, t])))

    n_panels = 1 + len(overlays)                   # original + one per word
    fig, axes = plt.subplots(1, n_panels, figsize=(3 * n_panels, 3.5))
    if n_panels == 1:
        axes = [axes]
    axes[0].imshow(image_denorm)
    axes[0].set_title(f"input\n{vis_ids[i]}", fontsize=8)
    axes[0].axis("off")
    for j, (token, alpha_map) in enumerate(overlays):
        ax = axes[j + 1]
        ax.imshow(image_denorm)
        ax.imshow(alpha_map, cmap="jet", alpha=0.5)
        ax.set_title(token, fontsize=10)
        ax.axis("off")

    caption_str = " ".join(t for t, _ in overlays)
    reference = test_captions_map[vis_ids[i]][0]
    fig.suptitle(f"pred: {caption_str}\nref:  {reference}", fontsize=9)
    plt.tight_layout()
    plt.show()

    if vis_run is not None:
        import wandb
        vis_run.log({f"attention/{vis_ids[i]}": wandb.Image(fig),
                     f"attention/{vis_ids[i]}/caption": caption_str,
                     f"attention/{vis_ids[i]}/reference": reference})

finish_wandb(vis_run)

For each visualized example: does the attention concentrate on plausibly relevant regions, or is it diffuse / off-target? Identify content words where the map matched a real object (success) and at least one where it did not (failure - hallucinated object, generic phrasing, repetition). Connect what you see in the maps back to the model's BLEU score and failure modes.

<span style="color: red;"><strong>TODO:</strong> Write your reflection on the attention visualizations here.</span>

## Additional Task 3: Pretrained GloVe Word Embeddings (bonus)

**Hypothesis.** Flickr8k's training set has only ~6,500 images / ~32k captions, which is a small signal for learning the structure of English from scratch in a `(vocab_size x EMBED_DIM)` lookup table. Initializing the decoder's word embedding with **pretrained GloVe vectors** (trained on ~6B tokens of Wikipedia + Gigaword) injects general semantic structure for free, so we expect:

- **Faster convergence** (lower train/val loss in early epochs),
- **Lower test perplexity** and **higher BLEU** vs the from-scratch Model 1 baseline,
- The benefit being concentrated on content words (nouns / verbs) that occur infrequently in our training captions.

**Setup.** We use **GloVe 6B 200d** (~250MB) for a closer match to Show and Tell's hidden sizes than the 50d/100d variants while staying smaller than 300d. We rebuild a `ShowAndTellCaptioner` with `EMBED_DIM = 200`, initialize the embedding matrix from GloVe for tokens that appear in both vocabularies (special tokens + OOV words get a small random init), and train with the exact same recipe (Adam, lr, weight decay, augmentation, early stopping) as Model 1. We report GloVe coverage and compare metrics against the 256d baseline; the embedding-dim difference is a known limitation we call out in the reflection.

In [ ]:
# Task 3 - GloVe-initialized Show and Tell.
import os, zipfile, urllib.request

GLOVE_DIM = 200
GLOVE_ZIP_URL = "https://nlp.stanford.edu/data/glove.6B.zip"  # ~822MB; contains 50/100/200/300d
GLOVE_DIR = "glove"
GLOVE_TXT = os.path.join(GLOVE_DIR, f"glove.6B.{GLOVE_DIM}d.txt")
GLOVE_ZIP = os.path.join(GLOVE_DIR, "glove.6B.zip")

# Download once per session (Colab) or once per machine (local). Skip if already on disk.
os.makedirs(GLOVE_DIR, exist_ok=True)
if not os.path.exists(GLOVE_TXT):
    if not os.path.exists(GLOVE_ZIP):
        print(f"Downloading GloVe from {GLOVE_ZIP_URL} (~822MB, one-time) ...")
        urllib.request.urlretrieve(GLOVE_ZIP_URL, GLOVE_ZIP)
    print(f"Extracting glove.6B.{GLOVE_DIM}d.txt ...")
    with zipfile.ZipFile(GLOVE_ZIP) as z:
        z.extract(f"glove.6B.{GLOVE_DIM}d.txt", GLOVE_DIR)

# Load only the vectors for tokens we actually have in our vocabulary - keeps RAM modest.
vocab_words = set(tokenizer.stoi.keys())
glove_vectors = {}
with open(GLOVE_TXT, "r", encoding="utf-8") as f:
    for line in f:
        word, _, rest = line.partition(" ")
        if word in vocab_words:
            glove_vectors[word] = np.fromstring(rest, sep=" ", dtype=np.float32)

# Build the embedding matrix aligned to our vocab. Specials and OOV get a small
# Gaussian init; padding_idx will be zeroed by nn.Embedding anyway.
embed_matrix = np.random.normal(0.0, 0.1, size=(len(tokenizer), GLOVE_DIM)).astype(np.float32)
n_found = 0
for word, idx in tokenizer.stoi.items():
    if word in tokenizer.special_tokens:
        continue
    vec = glove_vectors.get(word)
    if vec is not None:
        embed_matrix[idx] = vec
        n_found += 1
non_special_vocab = len(tokenizer) - len(tokenizer.special_tokens)
coverage = n_found / max(non_special_vocab, 1)
print(f"GloVe coverage: {n_found:,}/{non_special_vocab:,} non-special tokens ({coverage:.1%})")

# Build a Model 1 variant with EMBED_DIM = GLOVE_DIM and the GloVe-initialized embedding.
set_seed(SEED)
encoder_m1g = ShowAndTellEncoder(freeze=FREEZE_ENCODER).to(DEVICE)
model_m1g = ShowAndTellCaptioner(
    encoder=encoder_m1g,
    vocab_size=len(tokenizer),
    encoder_dim=ENCODER_DIM,
    embed_dim=GLOVE_DIM,
    decoder_dim=DECODER_DIM,
    dropout=DROPOUT,
    bos_idx=tokenizer.bos_idx,
    eos_idx=tokenizer.eos_idx,
    pad_idx=tokenizer.pad_idx,
).to(DEVICE)
with torch.no_grad():
    model_m1g.embedding.weight.copy_(torch.from_numpy(embed_matrix).to(DEVICE))
    model_m1g.embedding.weight[tokenizer.pad_idx].zero_()   # honor padding_idx

trainable_m1g = [p for p in model_m1g.parameters() if p.requires_grad]
print(f"trainable params: {sum(p.numel() for p in trainable_m1g):,}")

criterion_m1g = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_idx)
optimizer_m1g = torch.optim.Adam(trainable_m1g, lr=LR_M1, weight_decay=WEIGHT_DECAY_M1)

config_m1g = {
    **config_m1,
    "model": "show-and-tell+glove",
    "embed_dim": GLOVE_DIM,
    "embedding_init": "glove.6B.200d",
    "glove_coverage": coverage,
}
wandb_run_m1g = init_wandb("show-and-tell-glove", config_m1g, group="glove-comparison")

history_m1g = fit(
    model_m1g, train_loader, val_loader,
    criterion=criterion_m1g, optimizer=optimizer_m1g,
    num_epochs=NUM_EPOCHS_M1, patience=PATIENCE_M1,
    wandb_run=wandb_run_m1g,
)
plot_curves(history_m1g, title="Show and Tell + GloVe init - training")

metrics_m1g, hypotheses_m1g = evaluate_model(model_m1g, test_loader, tokenizer, criterion_m1g)
print("\nShow-and-Tell + GloVe test metrics:")
for k, v in metrics_m1g.items():
    print(f"  {k}: {v:.4f}")
if wandb_run_m1g is not None:
    wandb_run_m1g.log({f"test/{k}": v for k, v in metrics_m1g.items()})

show_predictions(model_m1g, test_loader, tokenizer, num_examples=4)
finish_wandb(wandb_run_m1g)

# Baseline vs GloVe comparison (note: baseline is 256d, GloVe variant is 200d).
print("\nBaseline (256d, scratch) vs GloVe (200d) on test:")
print(f"{'metric':<14}{'baseline':>14}{'+ GloVe':>14}{'delta':>10}")
for key in ("perplexity", "bleu1", "bleu2", "bleu3", "bleu4"):
    base, glove = metrics_m1[key], metrics_m1g[key]
    print(f"{key:<14}{base:>14.4f}{glove:>14.4f}{(glove - base):>+10.4f}")

Did GloVe initialization help, hurt, or have no effect on convergence speed (training/val loss curves) and on final test perplexity / BLEU? Connect the result to the GloVe coverage figure printed above and to the embedding-dim difference vs the 256d baseline. Are there qualitative caption changes (e.g. better use of rare nouns) that BLEU does not capture?

<span style="color: red;"><strong>TODO:</strong> Write your reflection on the GloVe experiment here.</span>